# Body assets setup — reconstructed
Restores the source download, conversion and schematic adaptation steps from our conversation. Existing original and adapted geometry are reused, not overwritten. No Hevy API access is needed. Requires NumPy; source downloads are needed only for missing files. Preserve the downloaded MIT notices.

In [1]:
%pip install numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json
import numpy as np
# Start with the original project; change only this line if needed.
project_folder = Path("C:/Users/AndyM/Projects/hevy")
assets_folder = project_folder / "assets"
data_folder = project_folder / "data"
print("Project:", project_folder)

Project: C:\Users\AndyM\Projects\hevy


## Load existing geometry, or download and convert the pinned source

In [3]:
import re
import urllib.request
source_folder = assets_folder / "source"
license_folder = assets_folder / "licenses"
for folder in [assets_folder, source_folder, license_folder]:
    folder.mkdir(parents=True, exist_ok=True)
commit = "28035c2c11d49038026e5c7e038c623f331edb60"
base_url = f"https://raw.githubusercontent.com/giavinh79/react-body-highlighter/{commit}/"
def cached_text(path, url):
    if path.exists():
        return path.read_text(encoding="utf-8")
    request = urllib.request.Request(url, headers={"User-Agent": "WorkoutHeatmapProject"})
    with urllib.request.urlopen(request, timeout=30) as response:
        text = response.read().decode("utf-8")
    path.write_text(text, encoding="utf-8")
    return text
original_path = assets_folder / "body_regions.json"
if original_path.exists():
    original_regions = json.loads(original_path.read_text(encoding="utf-8"))
    print("Using existing original geometry.")
else:
    geometry = cached_text(source_folder / "body_geometry.ts", base_url + "src/assets/index.ts")
    metadata = cached_text(source_folder / "body_metadata.ts", base_url + "src/component/metadata.ts")
    names = dict(re.findall(r"^\s*(\w+):\s*'([^']+)'", metadata, re.MULTILINE))
    def extract(side):
        match = re.search(rf"export const {side}Data[^=]*=\s*\[(.*?)\n\];", geometry, re.DOTALL)
        if match is None:
            raise ValueError(f"Missing source section {side}")
        regions = {}
        for name, text in re.findall(r"muscle:\s*MuscleType\.(\w+),\s*svgPoints:\s*\[(.*?)\]", match.group(1), re.DOTALL):
            shapes = []
            for coordinates in re.findall(r"'([^']+)'", text):
                values = [float(v) for v in coordinates.replace(',', ' ').split()]
                if len(values) < 6 or len(values) % 2:
                    raise ValueError("Invalid polygon")
                shapes.append(np.array(values).reshape(-1, 2).tolist())
            regions.setdefault(names[name], []).extend(shapes)
        return regions
    original_regions = {"front": extract("anterior"), "back": extract("posterior")}
    if len(original_regions["front"]) != 13 or len(original_regions["back"]) != 14:
        raise ValueError("Unexpected source region counts")
    original_path.write_text(json.dumps(original_regions, indent=2), encoding="utf-8")
# Retrieve notices only if missing; preserve existing copies.
cached_text(license_folder / "react-body-highlighter-LICENSE.txt", base_url + "LICENSE")
cached_text(license_folder / "react-native-body-highlighter-LICENSE.txt",
            "https://raw.githubusercontent.com/HichamELBSI/react-native-body-highlighter/master/LICENSE")
print("Original geometry and license notices available.")

Using existing original geometry.
Original geometry and license notices available.


## Polygon clipping
Intersections use p + t(q − p). The splits are visual approximations, not anatomical measurements.

In [4]:
def clip_polygon(points, axis, boundary, keep_lower):
    points = np.asarray(points, dtype=float)
    output = []
    def inside(point):
        return point[axis] <= boundary if keep_lower else point[axis] >= boundary
    previous = points[-1]
    for current in points:
        if inside(previous) != inside(current):
            t = (boundary - previous[axis]) / (current[axis] - previous[axis])
            output.append((previous + t * (current - previous)).tolist())
        if inside(current):
            output.append(current.tolist())
        previous = current
    return output

## Adapt labels and regions; preserve existing adapted asset

In [5]:
mapping = {"chest":"chest", "obliques":"abdominals", "abs":"abdominals",
 "biceps":"biceps", "triceps":"triceps", "neck":"neck",
 "front-deltoids":"shoulders", "back-deltoids":"shoulders", "forearm":"forearms",
 "trapezius":"traps", "lower-back":"lower_back", "hamstring":"hamstrings",
 "quadriceps":"quadriceps", "calves":"calves", "left-soleus":"calves",
 "right-soleus":"calves", "abductors":"adductors", "adductor":"adductors"}
expected = {"abdominals", "abductors", "adductors", "biceps", "calves", "chest", "forearms",
 "glutes", "hamstrings", "lats", "lower_back", "neck", "quadriceps", "shoulders", "traps", "triceps", "upper_back"}
adapted_path = assets_folder / "body_regions_hevy.json"
if adapted_path.exists():
    asset = json.loads(adapted_path.read_text(encoding="utf-8"))
    print("Preserving existing adapted geometry.")
else:
    adapted = {"front": {}, "back": {}}
    context = {"front": {}, "back": {}}
    def add(side, muscle, shapes):
        adapted[side].setdefault(muscle, []).extend(shapes)
    for side, regions in original_regions.items():
        for name, shapes in regions.items():
            if name in {"head", "knees"}:
                context[side][name] = shapes
            elif name == "upper-back":
                for shape in shapes:
                    add(side, "upper_back", [clip_polygon(shape, 1, 590, True)])
                    add(side, "lats", [clip_polygon(shape, 1, 590, False)])
            elif name == "gluteal":
                for shape in shapes:
                    left = np.mean([p[0] for p in shape]) < 500
                    boundary = 350 if left else 650
                    add(side, "abductors", [clip_polygon(shape, 0, boundary, left)])
                    add(side, "glutes", [clip_polygon(shape, 0, boundary, not left)])
            elif name in mapping:
                add(side, mapping[name], shapes)
            else:
                raise ValueError(f"Unmapped source region: {name}")
    asset = {"view_box": [0,0,1000,2000], "muscle_regions": adapted, "context_regions": context,
      "notes": ["Converted from react-body-highlighter; retain both MIT license notices.",
      "Schematic splits: upper back/lats at y=590; outer glutes used for abductors.",
      "Source front inner-thigh abductors relabeled adductors; abs/obliques combined.",
      "Illustrative boundaries, not precise anatomical measurements."]}
    mapped = set(adapted["front"]) | set(adapted["back"])
    if mapped != expected:
        raise ValueError(f"Unexpected mapping: {mapped ^ expected}")
    for regions in adapted.values():
        for shapes in regions.values():
            if any(len(shape) < 3 for shape in shapes):
                raise ValueError("Empty polygon after clipping")
    adapted_path.write_text(json.dumps(asset, indent=2), encoding="utf-8")
notes_path = assets_folder / "SOURCES_recovery.md"
notes_path.write_text(
    f"# Sources and adaptation\nGeometry: https://github.com/giavinh79/react-body-highlighter/tree/{commit}\n"
    "Upstream attribution: https://github.com/HichamELBSI/react-native-body-highlighter\n"
    "Both MIT notices retained in licenses/. Existing assets are preserved.\n"
    + "\n".join(asset.get("notes", [])), encoding="utf-8")
print("Adapted geometry:", adapted_path)

Preserving existing adapted geometry.
Adapted geometry: C:\Users\AndyM\Projects\hevy\assets\body_regions_hevy.json
